## test.py

In [3]:
import pandas as pd, joblib, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

TARGET = "price_range"

data = pd.read_csv("data.csv").dropna()
features = data.drop(columns=[TARGET])
encoder = LabelEncoder()
labels = encoder.fit_transform(data[TARGET])
num_classes = len(encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels)

# save test data (with the answer column) for the web app
test_file = X_test.copy()
test_file[TARGET] = encoder.inverse_transform(y_test)
test_file.to_csv("test_data.csv", index=False)

all_models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=42),
}

def score(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)
    avg = "binary" if num_classes == 2 else "weighted"
    if num_classes == 2:
        auc = roc_auc_score(y, proba[:, 1])
    else:
        auc = roc_auc_score(y, proba, multi_class="ovr", average="weighted")
    return [accuracy_score(y, pred), auc,
            precision_score(y, pred, average=avg, zero_division=0),
            recall_score(y, pred, average=avg, zero_division=0),
            f1_score(y, pred, average=avg, zero_division=0),
            matthews_corrcoef(y, pred)]

os.makedirs("model", exist_ok=True)
print("| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |")
print("|---|---|---|---|---|---|---|")
for name, clf in all_models.items():
    pipe = Pipeline([("scale", StandardScaler()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, "model/" + name.replace(" ", "_").lower() + ".joblib")
    r = score(pipe, X_test, y_test)
    print("| " + name + " | " + " | ".join(f"{v:.4f}" for v in r) + " |")

joblib.dump(encoder, "model/label_encoder.joblib")
joblib.dump(list(features.columns), "model/feature_columns.joblib")

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.9650 | 0.9987 | 0.9650 | 0.9650 | 0.9650 | 0.9534 |
| Decision Tree | 0.8300 | 0.8867 | 0.8319 | 0.8300 | 0.8302 | 0.7738 |
| KNN | 0.5000 | 0.7697 | 0.5211 | 0.5000 | 0.5054 | 0.3350 |
| Naive Bayes | 0.8100 | 0.9506 | 0.8113 | 0.8100 | 0.8105 | 0.7468 |


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


| Support Vector Machine | 0.8950 | 0.9833 | 0.8969 | 0.8950 | 0.8956 | 0.8602 |


['model/feature_columns.joblib']